# 판본 비교를 로컬 PostgreSQL로 옮기기 — 실험 노트북

SQL 연동 보고서 8장 "판본 비교도 SQL로 옮길 수 있나"의 실험을 한 노트북으로 정리한 것임.

**이 노트북이 하는 일**
- KR Notation Guide **2022판 ↔ 2026판**을 노테이션 단위로 비교해 *신설 · 삭제 · 명칭 변경 · 요건 변경*을 찾는 일(Part 1의 E12-4 "판 비교 처리")을 Python 대신 **PostgreSQL 안에서** 판정하게 옮김
- 옮긴 결과가 원래 Python 결과(93건)와 **바이트까지 같은지**, 그 표로 검색을 돌려도 보고서의 순위가 **그대로인지** 확인함

**원래 코드와의 관계**: 같은 폴더의 `run_experiment.py`(①–⑤), `edition_diff.sql`(DB 표 · 함수), `check_search.py`(⑥)를 한 노트북으로 합치고, 단계마다 셀을 나눠 주석을 단 것. 계산은 같음. 마지막 ⑦ 셀이 보고서에 쓴 결과 파일과 대조해 확인함

| 항목 | 내용 |
|---|---|
| 실행 위치 | 이 PC. Colab에서는 안 됨 (로컬 DB는 인터넷에서 접속할 수 없음) |
| 커널 | 저장소의 에이전트 백엔드 가상환경 `backend\.venv\Scripts\python.exe` (Python 3.13, psycopg · faiss · torch 포함). VS Code에서 "커널 선택 → Python 환경"으로 고름 |
| DB | 로컬 PostgreSQL 16.15 + pgvector 0.8.6, `localhost:5433`, 사용자 `postgres` (비밀번호 없음, 이 PC 안에서만 접속됨) |
| DB 설치 | 없으면 `localdb\restore_localdb.ps1`을 먼저 실행 (약 1분, 관리자 권한 필요 없음). 0-2 셀이 꺼진 서버는 켜 줌 |
| 필요한 파일 | Part 1 제출 폴더(`data\HD_RAG_Part1` 또는 `PART1_DIR`)의 `KR_Notation_2022.pkl` · `KR_Notation_2026.pkl` · `outputs\kr_edition_diff.csv`. `kr_edition_diff.py`(판본 비교 원본 스크립트)와 `final27_per_question.csv`(보고서 기준 값)는 이 폴더에 같이 있음 |
| 걸리는 시간 | ①–⑤ 약 10초, ⑥ 검색 확인 1–2분 (저장된 캐시만 써서 GPU가 필요 없음) |

**순서** (번호는 원래 스크립트의 번호를 그대로 씀. ④-1은 ③보다 먼저 돎)
```
① 원래 Python 판본 비교를 다시 돌려 기준 93건 확인
→ 파싱: 쪽 → 절, 노테이션 후보 (Python에 남기는 부분)
→ ② DB에 표 · 함수를 만들고 파싱 결과를 넣음
→ ④-1 비교용 정규화가 Python과 글자까지 같은지
→ ③ SQL 판정 세 가지(lev · trgm · seq)를 93건과 비교
→ ④-2 명칭 변경 후보 쌍의 유사도 비교
→ ⑤ 결과를 DB 표에 저장하고, 검색에 쓰는 모양이 같은지
→ ⑥ 비교표를 바꿔 끼우고 검색 (구성 4개 × 질문 37개)
→ ⑦ 보고서에 쓴 결과 파일과 대조
```

**보고서에 쓴 결과** (이 노트북이 다시 내야 하는 값)

| 판정 방법 | 건수 (신설 · 삭제 · 명칭 · 요건) | 기존 93건과 |
|---|---|---|
| Python (기준) | 93 (55 · 9 · 10 · 19) | 바이트까지 같음 |
| SQL lev (미리 정한 주 기준) | 94 (56 · 10 · 9 · 19) | 92건 같음. Chemical Tanker 명칭 변경 1건이 삭제 + 신설로 갈림 |
| SQL trgm (참고) | 100 (62 · 16 · 3 · 19) | 86건 같음. 명칭 변경 7건을 놓침 |
| SQL seq (Python과 같은 알고리즘) | 93 (55 · 9 · 10 · 19) | 바이트까지 같음 |

검색: 비교표를 무엇으로 바꿔도 구성 4개 × 37문항이 같음. 비교표를 비우면(대조군) 최종 통합 test nDCG@10이 0.988 → 0.893

> ⚠️ 이 노트북은 `hdrag` DB의 표를 **지우고 다시 만듦**. 지금 DB 내용은 `localdb\hdrag.dump`에 백업되어 있음

In [1]:
# 0-1. 설정: 경로 · DB 접속 정보 · 결과 폴더
# ─────────────────────────────────────────────────────────────────────────────
# 경로는 저장소 안에서 찾는다. Part 1 데이터 위치만 PART1_DIR(환경 변수 또는 backend\.env)로 바꿀 수 있다.
import csv              # 판본 비교 결과(CSV) 읽고 쓰기
import hashlib          # 파일이 바이트까지 같은지 볼 때 쓰는 md5
import importlib.util   # 경로로 .py 파일을 모듈처럼 불러오기 (kr_edition_diff.py)
import json             # 요약 · 검색 확인 결과 저장
import os
import subprocess       # 꺼진 로컬 PostgreSQL 서버 켜기 (pg_ctl)
import sys
import time             # 단계별 걸린 시간
from collections import defaultdict
from difflib import SequenceMatcher   # Python 쪽 문자열 유사도 (명칭 변경 판정의 원래 방법)
from pathlib import Path

import psycopg          # PostgreSQL 드라이버 (psycopg 3)

# 이 노트북 · 실험 파일 · 보고서에 쓴 결과 파일이 있는 폴더 (저장소의 db\edition_diff).
# Jupyter · VS Code는 노트북이 있는 폴더에서 커널을 켜므로 지금 폴더를 씀
NB_DIR = Path.cwd()
if not (NB_DIR / "edition_diff.sql").exists():
    raise SystemExit(f"노트북 폴더(db\\edition_diff)에서 실행해야 함. 지금 폴더: {NB_DIR}")
REPO = NB_DIR.parents[1]   # hd-rag-part2 저장소 맨 위
# 에이전트 백엔드 (⑥ 검색 확인에서 검색기를 불러옴)
BACKEND = REPO / "backend"


def env_value(name):
    """환경 변수 → backend\.env 순으로 설정 값을 찾음 (백엔드와 같은 설정을 씀)"""
    if os.environ.get(name):
        return os.environ[name]
    env = BACKEND / ".env"
    if env.exists():
        for line in env.read_text(encoding="utf-8-sig").splitlines():
            if line.startswith(name + "="):
                return line.split("=", 1)[1].strip() or None
    return None


# Part 1 제출 폴더: KR 2022 · 2026판 pkl(과제 원본 텍스트)과 기준 결과 outputs\kr_edition_diff.csv
# 백엔드(rag/part1.py)와 같은 순서로 찾음: PART1_DIR → 저장소 data\HD_RAG_Part1 → 만든 사람 PC의 경로
PART1 = next((Path(c) for c in (env_value("PART1_DIR"), REPO / "data" / "HD_RAG_Part1",
                                 r"C:\Users\710pk\OneDrive\Desktop\HD융캡디\제출\HD_RAG_Part1")
              if c and (Path(c) / "test_set.csv").exists()), REPO / "data" / "HD_RAG_Part1")
REF = PART1 / "outputs" / "kr_edition_diff.csv"   # Python 판본 비교 결과 93건 (검색이 실제로 쓰는 표)
# 원래 Python 판본 비교 스크립트 (파싱 함수를 그대로 빌려 씀). Part 1의 07_판비교_쿼리손질 폴더에서 복사해 둔 것
SRC = NB_DIR / "kr_edition_diff.py"
# 보고서의 문항별 기준 값 (구성 4개 × 37문항의 nDCG@10과 정답 쪽 순위). 실험 노트북 결과를 복사해 둔 것
FINAL27 = NB_DIR / "final27_per_question.csv"

# 이 노트북의 결과는 새 하위 폴더에 씀 → 보고서에 쓴 결과 파일(NB_DIR 바로 아래)을 덮어쓰지 않고 ⑦에서 대조함
OUT = NB_DIR / "notebook_out"
OUT.mkdir(exist_ok=True)

# 로컬 PostgreSQL 접속 정보. 이 PC 안에서만 접속을 받는 서버라 비밀번호가 없음 (initdb -A trust)
PG = dict(host="localhost", port=5433, user="postgres")
DBNAME = "hdrag"        # 실험용 DB 이름
# 서버 프로그램 · 데이터 폴더 위치 (restore_localdb.ps1의 기본 설치 위치). 환경 변수 LOCALPG_DIR로 바꿀 수 있음
LOCALPG = Path(os.environ.get("LOCALPG_DIR") or Path.home() / "dev" / "localpg")

# 판본 비교 CSV의 머리글. Python 스크립트와 이름 · 순서가 같아야 바이트 비교가 됨
HEAD = ["절", "구분", "노테이션", "2022 쪽", "2026 쪽", "2022 요건", "2026 요건"]


def md5(path):
    """파일 내용의 md5. 두 파일이 바이트까지 같은지 볼 때 씀"""
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def read_rows(path):
    """판본 비교 CSV → 머리글을 뺀 행 튜플 목록 (utf-8-sig: 엑셀용 BOM이 붙어 있어서)"""
    with open(path, encoding="utf-8-sig") as f:
        return [tuple(r) for r in list(csv.reader(f))[1:]]


summary = {}   # 단계별 숫자를 모아 ⑤ 끝에서 summary.json으로 저장 (원래 스크립트와 같은 키)

for p in (NB_DIR, PART1, REF, SRC, BACKEND, FINAL27):
    print("있음 " if p.exists() else "없음!", p)

있음  C:\Users\710pk\dev\hd-rag-part2\db\edition_diff
있음  C:\Users\710pk\OneDrive\Desktop\HD융캡디\제출\HD_RAG_Part1
있음  C:\Users\710pk\OneDrive\Desktop\HD융캡디\제출\HD_RAG_Part1\outputs\kr_edition_diff.csv
있음  C:\Users\710pk\dev\hd-rag-part2\db\edition_diff\kr_edition_diff.py
있음  C:\Users\710pk\dev\hd-rag-part2\backend
있음  C:\Users\710pk\dev\hd-rag-part2\db\edition_diff\final27_per_question.csv


<>:29: SyntaxWarning: invalid escape sequence '\.'
<>:29: SyntaxWarning: invalid escape sequence '\.'
C:\Users\710pk\AppData\Local\Temp\ipykernel_35248\301415022.py:29: SyntaxWarning: invalid escape sequence '\.'
  """환경 변수 → backend\.env 순으로 설정 값을 찾음 (백엔드와 같은 설정을 씀)"""


In [2]:
# 0-2. 로컬 PostgreSQL 서버가 켜져 있는지 확인하고, 꺼져 있으면 켬
# ─────────────────────────────────────────────────────────────────────────────
# PC를 껐다 켜면 서버도 꺼져 있다. 접속해 보고, 안 되면 pg_ctl로 켠다.
def pg_alive():
    """postgres DB에 접속이 되면 True (= 서버가 켜져 있음)"""
    try:
        with psycopg.connect(**PG, dbname="postgres", connect_timeout=3):
            return True
    except psycopg.OperationalError:
        return False


if not pg_alive():
    bin_dir = LOCALPG / "env" / "Library" / "bin"   # conda-forge 패키지는 Windows에서 실행 파일을 Library\bin에 둠
    data_dir = LOCALPG / "data"                      # initdb로 만든 데이터 폴더 (DB 파일이 실제로 있는 곳)
    if not (bin_dir / "pg_ctl.exe").exists():
        raise SystemExit(f"로컬 PostgreSQL이 없음: {LOCALPG}\n→ {NB_DIR / 'localdb' / 'restore_localdb.ps1'}을 먼저 실행")
    # pg_ctl을 이 커널과 떨어진 프로세스로 띄움. 보통 방식으로 띄우면 서버가 커널의 출력 통로를 물고 있어서
    # 셀이 끝나지 않는 일이 있음 → DETACHED_PROCESS로 분리하고, 서버 기록은 pg.log 파일로 보냄
    subprocess.Popen([str(bin_dir / "pg_ctl.exe"), "-D", str(data_dir), "-l", str(LOCALPG / "pg.log"),
                      "-o", f"-p {PG['port']}", "start"],
                     creationflags=subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP,
                     stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):                              # 켜질 때까지 최대 30초 기다림
        time.sleep(1)
        if pg_alive():
            break
    else:
        raise SystemExit(f"서버가 켜지지 않음. {LOCALPG / 'pg.log'} 확인")

with psycopg.connect(**PG, dbname="postgres") as c:
    print(c.execute("SELECT version()").fetchone()[0].split(",")[0], "· 포트", PG["port"])
    # pgvector가 설치돼 있는지 확인. 판본 비교는 벡터를 쓰지 않지만, 같은 서버에 청크 벡터도 넣을 수 있음
    print("pgvector:", c.execute("SELECT default_version FROM pg_available_extensions WHERE name = 'vector'").fetchone()[0])

PostgreSQL 16.15 · 포트 5433
pgvector: 0.8.6


## ① 기준: 원래 Python 판본 비교를 다시 돌림

Part 1에서 만든 `kr_edition_diff.py`의 판정 규칙 (KR 가이드에는 ABS 같은 개정 이력 페이지가 없어서 두 판을 직접 맞대 봄)

| 판정 | 규칙 |
|---|---|
| 노테이션 후보 | 2장 각 절의 NOTATIONS 목록 항목 · REQUIREMENTS 표의 행, 3 · 4장 "노테이션 \| 요건" 표의 행 |
| 절 맞추기 | 번호를 뺀 절 제목으로 짝지음 (판마다 번호 형식이 다름). 제목이 없는 이어지는 쪽은 앞 쪽의 절을 물려받음 |
| 신설 | 2026판 후보인데, 2022판의 같은 절 본문(그 판에 절이 없으면 문서 전체)에 이름이 없음 |
| 삭제 | 2022판 후보인데, 2026판의 같은 절 본문에 이름이 없음 |
| 명칭 변경 | 같은 절의 삭제 · 신설 쌍 중 이름 유사도(difflib SequenceMatcher)가 0.6 이상 |
| 요건 변경 | 이름은 같고 두 판 모두 요건(설계 · 검사 근거)이 있는데 그 문자열이 다름 |

- "이름이 본문에 있나"는 띄어쓰기 · 기호 · 대소문자를 무시하고 봄 (`compact`: 영문 소문자 · 숫자 · 한글만 남김)
- 이 스크립트가 지금도 93건을 그대로 내는지부터 확인해야, 뒤의 SQL 결과와 비교할 기준이 생김

In [3]:
# ① 원래 Python 판본 비교를 다시 돌려, 검색에 쓰는 93건(REF)과 바이트까지 같은지 확인
# ─────────────────────────────────────────────────────────────────────────────
# kr_edition_diff.py를 파일 경로로 바로 불러와 모듈 K로 씀 (07 폴더를 import 경로에 넣지 않으려고)
spec = importlib.util.spec_from_file_location("ked", SRC)
K = importlib.util.module_from_spec(spec)
spec.loader.exec_module(K)        # 모듈 본문 실행 → K.load_pages, K.compact 같은 함수가 생김 (main()은 아직 안 돎)

K.DATA = PART1                    # pkl을 읽을 폴더. 스크립트의 기본값(03_데이터)은 폴더 정리로 없어져서 바꿔 줌
K.OUT = OUT / "python_kr_edition_diff.csv"   # 결과를 노트북 폴더에 씀 → 07 폴더의 원본 CSV를 덮어쓰지 않음

t = time.perf_counter()
K.main()                          # 판본 비교 전체: pkl 읽기 → 파싱 → 판정 → CSV 저장 (절별 건수를 출력함)
summary["python_main_s"] = round(time.perf_counter() - t, 3)

ref_rows = read_rows(REF)         # 기준 93건
summary["ref_rows"] = len(ref_rows)
summary["python_same_bytes"] = md5(K.OUT) == md5(REF)   # 줄 순서 · 따옴표 · BOM까지 같은지
print(f"\n① Python 재실행: {len(read_rows(K.OUT))}건 · 기준 CSV와 바이트까지 같음 = {summary['python_same_bytes']}"
      f" ({summary['python_main_s']}초)")

절: 2022판 43개, 2026판 45개 | 노테이션 후보: 2022판 556개, 2026판 596개
변경 기록 93건 → python_kr_edition_diff.csv
  Ch 3                                               명칭 2, 삭제 3, 신설 24, 요건 10
  Ch 4                                               삭제 3, 신설 11, 요건 5
  Ore/Chemical Carrier                               명칭 2, 신설 2
  Bulk Carrier                                       요건 3
  Container Ship                                     명칭 1, 신설 2
  Liquefied Gas Carrier                              명칭 1, 삭제 1, 요건 1
  Oil/Chemical Tanker                                명칭 1, 삭제 1, 신설 1
  Cargo Ship                                         신설 2
  Chemical Tanker                                    명칭 2
  Mobile Offshore Unit                               신설 2
  Offshore Support Vessel                            신설 2
  Oil/Bulk/Ore Carrier                               신설 2
  Oil/Liquefied Gas Carrier                          명칭 1, 삭제 1
  Ore Carrier                                        신설 2
  Ore/Oil Carrie

## 파싱: DB에 넣을 재료 만들기

판정은 SQL로 옮기지만 **파싱은 Python에 둠**. 판마다 서식이 조금씩 달라 정규식 규칙이 많이 필요하고, 비교의 정확도를 대부분 이 부분이 정하기 때문임 (보고서 8.6)

| 변수 | 만드는 함수 (kr_edition_diff.py) | 내용 | 넣을 DB 표 |
|---|---|---|---|
| `P[연도]` | `load_pages` | 쪽 번호 → 쪽 텍스트 (pkl의 page_content) | `pages.raw` |
| `S[연도]` | `section_of_pages` | 쪽 번호 → 절 이름 (부록 · 목차 · 요약표 쪽은 빈 문자열) | `pages.section` |
| `U[연도]` | `page_units` → `collapse` | (절, 노테이션 이름) → (나온 쪽 목록, 요건 또는 None) | `notations` |

In [4]:
# 파싱: 쪽 텍스트 · 쪽별 절 · 노테이션 후보를 만든다 (모두 kr_edition_diff.py의 함수 그대로)
# ─────────────────────────────────────────────────────────────────────────────
YEARS = (2022, 2026)
# P[연도] = {쪽 번호(1부터): 쪽 텍스트}
#   pkl은 LangChain Document 목록인데, langchain 없이 읽으려고 스크립트의 SafeUnpickler가
#   모르는 클래스를 빈 껍데기 클래스로 바꿔 읽는다
P = {y: K.load_pages(f"KR_Notation_{y}.pkl") for y in YEARS}
# S[연도] = {쪽 번호: 절 이름}
#   쪽 위쪽의 "11. Container Ship" 같은 제목을 찾아 절 이름으로 씀(번호는 뺌). 제목이 없는 이어지는 쪽은 앞 쪽의 절을
#   물려받음. 부록(Annex) · 목차 · 요약표 쪽은 ''(비교하지 않음). 3 · 4장은 표 형식이 같아 "Ch 3" · "Ch 4"로 묶음
S = {y: K.section_of_pages(P[y]) for y in YEARS}
# U[연도] = {(절, 이름): (나온 쪽 목록, 요건 문자열 또는 None)}
#   page_units: 첫 칸 머리글에 "Notation"이 들어간 표의 행 → (이름 = 첫 칸, 요건 = 나머지 칸)
#               2장 NOTATIONS 목록의 항목 → (이름, 요건 없음)
#   collapse:   같은 (절, 이름)이 여러 쪽 · 여러 번 나오면 쪽은 모으고, 요건은 처음 나온 것 하나만 씀
U = {y: K.collapse(K.page_units(P[y], S[y])) for y in YEARS}

# PostgreSQL의 text 칸에는 NUL 문자(\x00)를 넣을 수 없음 → 있으면 여기서 멈춤 (KR 쪽에는 없음)
nul = sum("\x00" in t for y in P for t in P[y].values())
assert nul == 0, f"NUL 문자가 든 쪽 {nul}개"

for y in YEARS:
    secs = {s for s in S[y].values() if s}          # 쪽에 붙은 절 이름 전부
    secs_u = {s for s, _ in U[y]}                    # 그중 노테이션 후보가 하나라도 나온 절 (① 출력의 "절" 수)
    print(f"KR {y}: 쪽 {len(P[y])}개 · 절이 붙은 쪽 {sum(1 for s in S[y].values() if s)}개 · 절 {len(secs)}개"
          f" (노테이션 후보가 있는 절 {len(secs_u)}개) · 노테이션 후보 {len(U[y])}개")

KR 2022: 쪽 244개 · 절이 붙은 쪽 165개 · 절 45개 (노테이션 후보가 있는 절 43개) · 노테이션 후보 556개
KR 2026: 쪽 251개 · 절이 붙은 쪽 169개 · 절 47개 (노테이션 후보가 있는 절 45개) · 노테이션 후보 596개


## ② DB에 판본 비교 장치 만들기

아래 셀들은 원래 `edition_diff.sql` 한 파일을 단계별로 나눈 것임 (SQL 문장은 같고 주석만 더함). 만드는 것:

| 종류 | 이름 | 하는 일 |
|---|---|---|
| 표 | `documents` | 판 목록 (KR2022, KR2026) |
| 표 | `pages` | 쪽 원문 + Python이 정한 절 이름 |
| 표 | `notations` | 노테이션 후보 (판, 절, 이름, 나온 쪽, 요건) |
| 함수 | `kkey()` · `req_key()` | 비교용 정규화 (Python의 compact · req와 같게) |
| 구체화 뷰 | `section_texts` · `doc_texts` | 판 · 절별 본문, 판 전체 본문 (열쇠 형태로 이어 붙여 저장) |
| 함수 | `absent_in()` | 상대 판 본문에 이름이 없는지 |
| 함수 | `seq_ratio()` | Python difflib SequenceMatcher.ratio()와 같은 유사도 |
| 함수 | `name_sim()` | 유사도 세 가지(lev · trgm · seq) 중 하나 |
| 함수 | `edition_diff()` | 판본 비교 본체: 신설 · 삭제 · 명칭 변경 · 요건 변경 목록을 돌려줌 |
| 표 · 뷰 | `edition_changes` · `edition_change_pages` | 비교 결과 저장, 검색에 쓰는 모양(쪽 → 개정 종류) |

새 판(예: KR2027)이 오면: 파싱 결과를 pages · notations에 넣음 → 본문 뷰 REFRESH → `edition_diff('KR2026', 'KR2027', 0.6, 'seq')` 결과를 edition_changes에 INSERT

In [5]:
# ② hdrag DB가 없으면 만들고, 이 노트북 끝까지 쓸 연결을 엶
# ─────────────────────────────────────────────────────────────────────────────
# CREATE DATABASE는 트랜잭션 안에서 실행할 수 없어서 autocommit으로 접속함
with psycopg.connect(**PG, dbname="postgres", autocommit=True) as c:
    if not c.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DBNAME,)).fetchone():
        c.execute(f"CREATE DATABASE {DBNAME}")
        print(f"{DBNAME} DB를 새로 만듦")
# autocommit = 문장마다 바로 반영 (실험이라 되돌릴 일이 없고, 셀을 다시 실행하기 편함)
db = psycopg.connect(**PG, dbname=DBNAME, autocommit=True)
summary["server"] = db.execute("SELECT version()").fetchone()[0].split(",")[0]
print("접속:", summary["server"], "·", DBNAME)

접속: PostgreSQL 16.15 · hdrag


In [6]:
# ② SQL ⑴ 확장 기능 켜기 · 이전 실험 지우기 · 표 3개 만들기
# ─────────────────────────────────────────────────────────────────────────────
db.execute(r"""
-- fuzzystrmatch: levenshtein() 편집 거리 → 명칭 변경 'lev' 방법
-- pg_trgm:       similarity() 삼중 문자 유사도 → 명칭 변경 'trgm' 방법
CREATE EXTENSION IF NOT EXISTS fuzzystrmatch;
CREATE EXTENSION IF NOT EXISTS pg_trgm;

-- 다시 실행해도 되게 이전 실험의 함수 · 뷰 · 표를 지움 (CASCADE: 그것에 기대는 뷰 등도 함께 지움)
DROP FUNCTION IF EXISTS edition_diff, name_sim, seq_ratio, absent_in CASCADE;
DROP MATERIALIZED VIEW IF EXISTS doc_texts, section_texts CASCADE;
DROP TABLE IF EXISTS edition_changes, notations, pages, documents CASCADE;

-- 판 목록: 판 하나가 한 줄
CREATE TABLE documents (
    doc_id  text PRIMARY KEY,                       -- 'KR2022', 'KR2026'
    society text NOT NULL,                          -- 선급 'KR'
    year    int  NOT NULL,                          -- 판 연도
    UNIQUE (society, year)                          -- 같은 판을 다른 이름으로 두 번 넣지 못하게
);

-- 쪽: 쪽 하나가 한 줄. 판정에 쓰는 본문은 raw를 kkey()로 바꿔 만듦
CREATE TABLE pages (
    page_id text PRIMARY KEY,                       -- 'KR2026:176' (판:쪽 번호. 검색 결과 · 정답표와 같은 형식)
    doc_id  text NOT NULL REFERENCES documents,     -- 어느 판의 쪽인지 (documents에 없는 판이면 거부)
    page_no int  NOT NULL,                          -- 쪽 번호 (1부터, PDF의 몇 번째 장인지)
    raw     text NOT NULL,                          -- pkl의 page_content 그대로
    section text NOT NULL DEFAULT '',               -- Python 파싱 결과 (절이 없으면 '')
    UNIQUE (doc_id, page_no)
);

-- 노테이션 후보: (판, 절, 이름)이 한 줄
CREATE TABLE notations (                            -- NOTATIONS 목록 항목 · 요건표 행
    doc_id  text  NOT NULL REFERENCES documents,
    section text  NOT NULL,
    name    text  NOT NULL,                         -- 표시용 정규화(norm)까지 한 이름
    pages   int[] NOT NULL,                         -- 나온 쪽 (오름차순 배열)
    req     text,                                   -- 요건(설계 · 검사 근거). 목록 항목뿐이면 NULL
    PRIMARY KEY (doc_id, section, name)
);
""")
print("표:", [r[0] for r in db.execute("SELECT tablename FROM pg_tables WHERE schemaname = 'public' ORDER BY 1")])

표: ['agent_feedback', 'agent_runs', 'documents', 'entities', 'notations', 'pages']


In [7]:
# ② SQL ⑵ 비교용 정규화 함수 kkey() · req_key()
# ─────────────────────────────────────────────────────────────────────────────
# "이름이 본문에 있나"를 볼 때 띄어쓰기 · 괄호 · 대소문자 · 각주 표시 차이를 무시하려고, 이름과 본문을 모두
# 같은 '열쇠' 형태로 바꿔 비교함. Python의 compact()와 글자까지 같아야 함 (④-1에서 확인)
db.execute(r"""
-- kkey: 각주 위첨자(¹⁾ 등) 삭제 → NFKC 정규화(전각 · 합자 문자를 보통 문자로) → <br>을 공백으로 → 소문자
--       → 영문 · 숫자 · 한글만 남김
--   Python norm()이 따로 하는 '**' 제거 · 대시 통일 · 공백 정리는, 마지막 단계에서 어차피 지워지는 문자라 생략함
--   IMMUTABLE: 같은 입력이면 항상 같은 결과 → 구체화 뷰 · 색인에 쓸 수 있음
CREATE OR REPLACE FUNCTION kkey(s text) RETURNS text
LANGUAGE sql IMMUTABLE PARALLEL SAFE AS $$
    SELECT regexp_replace(
             lower(replace(normalize(regexp_replace(coalesce(s, ''), '[¹²³⁴⁵⁶⁷⁸⁹⁰]+[⁾)]?', '', 'g'), NFKC), '<br>', ' ')),
             '[^0-9a-z가-힣]', '', 'g')
$$;

-- req_key: 요건 비교용 열쇠. 2022판에는 각주 번호가 위첨자가 아닌 일반 문자("Pt 3, 2), 3)")로 남아 있어서
--   쉼표 · 공백 뒤의 '숫자)'를 먼저 지운 뒤 kkey (Python kr_edition_diff의 req()와 같음)
CREATE OR REPLACE FUNCTION req_key(s text) RETURNS text
LANGUAGE sql IMMUTABLE PARALLEL SAFE AS $$
    SELECT kkey(regexp_replace(coalesce(s, ''), '(?<=[,\s])\d\)', '', 'g'))
$$;
""")
# 예: 이름의 각주 · 괄호 · 쉼표 · 대소문자가 사라지고, 요건의 일반 문자 각주 번호 "2), 3)"이 사라짐
print(db.execute("SELECT kkey('LS(CL, RS)¹⁾'), req_key('Pt 3, 2), 3) Annex 3-2')").fetchone())
print("Python compact:", K.compact("LS(CL, RS)¹⁾"))

('lsclrs', 'pt3annex32')
Python compact: lsclrs


In [8]:
# ② SQL ⑶ 절 본문 · 판 본문 구체화 뷰
# ─────────────────────────────────────────────────────────────────────────────
# 구체화 뷰(materialized view) = 쿼리 결과를 표처럼 저장해 두는 뷰. 판정 때마다 본문을 다시 이어 붙이지 않으려고 씀
# 지금은 pages가 비어 있어 뷰도 비어 있음 → 데이터를 넣은 뒤(② 마지막 셀) REFRESH로 다시 계산함
db.execute(r"""
-- section_texts: (판, 절)마다 그 절에 속한 쪽들의 열쇠를 쪽 순서대로 이어 붙인 '절 본문'
--   absent_in()이 "이 이름이 상대 판의 같은 절에 있나"를 strpos 한 번으로 보게 하려는 것
CREATE MATERIALIZED VIEW section_texts AS
    SELECT doc_id, section, string_agg(kkey(raw), '' ORDER BY page_no) AS hay
    FROM pages WHERE section <> '' GROUP BY doc_id, section;
CREATE UNIQUE INDEX ON section_texts (doc_id, section);   -- (판, 절)로 바로 찾기

-- doc_texts: 판 전체 본문. 상대 판에 같은 절이 없을 때(절 신설 · 절 삭제) 문서 전체에서 찾으려고
CREATE MATERIALIZED VIEW doc_texts AS
    SELECT doc_id, string_agg(kkey(raw), '' ORDER BY page_no) AS hay
    FROM pages GROUP BY doc_id;
CREATE UNIQUE INDEX ON doc_texts (doc_id);
""")
print("구체화 뷰:", [r[0] for r in db.execute("SELECT matviewname FROM pg_matviews ORDER BY 1")])

구체화 뷰: ['doc_texts', 'section_texts']


In [9]:
# ② SQL ⑷ absent_in(): 상대 판 본문에 이름이 없는지 (신설 · 삭제 판정의 핵심)
# ─────────────────────────────────────────────────────────────────────────────
# 신설 = 새 판 후보 중 absent_in(이름, 절, 옛 판),  삭제 = 옛 판 후보 중 absent_in(이름, 절, 새 판)
# 구조(표 · 목록)끼리 맞대지 않고 본문에 글자가 있는지로 보는 이유: 판마다 파싱 서식이 조금씩 달라서,
# 구조끼리 맞대면 같은 노테이션도 '삭제 + 신설'로 잘못 나오기 쉬움
db.execute(r"""
-- absent_in(이름, 절, 판): 그 판의 같은 절 본문에 이름(열쇠)이 없으면 true
--   nullif(절 본문, ''): 그 판에 그 절이 없으면 NULL → coalesce가 판 전체 본문으로 대신함
--   strpos(본문, 이름) = 0 → 본문 안에서 이름을 찾지 못함
--   STABLE: 한 실행 안에서는 결과가 같음 (뷰를 읽으므로 IMMUTABLE은 아님)
CREATE OR REPLACE FUNCTION absent_in(p_name text, p_section text, p_doc text) RETURNS boolean
LANGUAGE sql STABLE AS $$
    SELECT strpos(coalesce(nullif((SELECT hay FROM section_texts WHERE doc_id = p_doc AND section = p_section), ''),
                           (SELECT hay FROM doc_texts WHERE doc_id = p_doc)),
                  kkey(p_name)) = 0
$$;
""")
print("absent_in() 만듦")

absent_in() 만듦


In [10]:
# ② SQL ⑸ seq_ratio(): Python difflib.SequenceMatcher(None, x, y).ratio()를 PL/pgSQL로 옮긴 것
# ─────────────────────────────────────────────────────────────────────────────
# 왜 만들었나: 미리 정한 lev(편집 거리)와 참고용 trgm으로는 Python 판정과 경계 사례가 갈려서(보고서 8.4),
#             Python과 같은 알고리즘을 그대로 옮김 (lev 결과를 본 뒤 추가한 것이라 보고서에 '사후 추가'로 적음)
# 계산 (Ratcliff/Obershelp):
#   1. 두 문자열에서 가장 긴 공통 구간(연속으로 같은 글자들)을 찾음
#   2. 그 구간의 왼쪽끼리, 오른쪽끼리 다시 1을 되풀이함
#   3. 찾은 공통 글자 수의 합을 m이라 하면  ratio = 2 × m ÷ (x 길이 + y 길이)
#   길이가 같은 공통 구간이 여럿이면 x에서 먼저 나온 것, 그다음 y에서 먼저 나온 것을 고름 (Python과 같은 동점 처리)
# 차이: y가 200자 이상이면 Python은 자주 나오는 글자를 버리는 autojunk를 써서 값이 달라질 수 있음
#       (여기 이름 열쇠는 모두 200자 미만임을 ③에서 assert로 확인)
db.execute(r"""
CREATE OR REPLACE FUNCTION seq_ratio(x text, y text) RETURNS double precision
LANGUAGE plpgsql IMMUTABLE AS $$
DECLARE
    a text[] := regexp_split_to_array(x, '');   -- x를 글자 배열로 (PostgreSQL 배열은 1부터 셈)
    b text[] := regexp_split_to_array(y, '');   -- y를 글자 배열로
    la int := length(x);
    lb int := length(y);
    stack int[];                 -- 아직 볼 구간들: (alo, ahi, blo, bhi) 네 개씩. 0부터 세고 끝(hi)은 포함 안 함
    alo int; ahi int; blo int; bhi int;          -- 지금 보는 구간 a[alo:ahi], b[blo:bhi]
    prev int[]; cur int[];       -- 동적 계획법 한 줄: cur[j+1] = a[i]와 b[j]에서 끝나는 공통 구간의 길이
    i int; j int; k int;
    bi int; bj int; bk int;      -- 지금까지 가장 긴 공통 구간: a에서의 시작 bi, b에서의 시작 bj, 길이 bk
    m int := 0;                  -- 찾은 공통 글자 수의 합
BEGIN
    IF la + lb = 0 THEN RETURN 1; END IF;       -- 둘 다 빈 문자열이면 Python도 1.0
    IF la = 0 OR lb = 0 THEN RETURN 0; END IF;
    stack := ARRAY[0, la, 0, lb];               -- 처음에는 전체 구간 하나
    WHILE coalesce(array_length(stack, 1), 0) > 0 LOOP
        -- 스택에서 구간 하나를 꺼냄
        alo := stack[1]; ahi := stack[2]; blo := stack[3]; bhi := stack[4];
        stack := stack[5:];
        -- 이 구간 안에서 가장 긴 공통 구간 찾기
        bi := alo; bj := blo; bk := 0;
        prev := array_fill(0, ARRAY[lb + 1]);
        FOR i IN alo .. ahi - 1 LOOP
            cur := array_fill(0, ARRAY[lb + 1]);
            FOR j IN blo .. bhi - 1 LOOP
                IF a[i + 1] = b[j + 1] THEN
                    k := CASE WHEN j > blo THEN prev[j] ELSE 0 END + 1;   -- prev[j] = (i−1, j−1)에서 끝나는 길이
                    cur[j + 1] := k;
                    -- 더 길 때만 바꿈 (같은 길이면 먼저 찾은 것을 둠 = Python과 같은 동점 처리)
                    IF k > bk THEN bi := i - k + 1; bj := j - k + 1; bk := k; END IF;
                END IF;
            END LOOP;
            prev := cur;
        END LOOP;
        IF bk > 0 THEN
            m := m + bk;
            -- 찾은 구간의 왼쪽 부분 · 오른쪽 부분을 다시 볼 구간으로 넣음 (양쪽 모두 비어 있지 않을 때만)
            IF alo < bi AND blo < bj THEN stack := stack || ARRAY[alo, bi, blo, bj]; END IF;
            IF bi + bk < ahi AND bj + bk < bhi THEN stack := stack || ARRAY[bi + bk, ahi, bj + bk, bhi]; END IF;
        END IF;
    END LOOP;
    RETURN 2.0 * m / (la + lb);
END
$$;
""")
# 확인: 몇 쌍을 Python SequenceMatcher와 나란히 계산 (아래 두 줄의 값이 같아야 함)
pairs_demo = [("abcd", "bcde"), ("lfss", "lffs"), ("chemicaltanker", "chemicaltankerapparentsg")]
print("SQL   ", [round(db.execute("SELECT seq_ratio(%s, %s)", p).fetchone()[0], 4) for p in pairs_demo])
print("Python", [round(SequenceMatcher(None, *p).ratio(), 4) for p in pairs_demo])

SQL    [0.75, 0.75, 0.7368]
Python [0.75, 0.75, 0.7368]


In [11]:
# ② SQL ⑹ name_sim(): 두 노테이션 이름의 유사도 (명칭 변경 판정에 씀)
# ─────────────────────────────────────────────────────────────────────────────
db.execute(r"""
-- name_sim(a, b, 방법): 0–1 사이 값. 모두 열쇠(kkey)끼리 비교함
--   'lev'  = 1 − 편집 거리 ÷ 긴 쪽 길이        (미리 정한 주 기준)
--   'trgm' = pg_trgm similarity: 세 글자씩 자른 조각이 얼마나 겹치나   (참고)
--   'seq'  = seq_ratio: Python SequenceMatcher와 같은 값             (lev 결과를 본 뒤 추가)
--   greatest(…, 1): 둘 다 빈 문자열일 때 0으로 나누지 않게
CREATE OR REPLACE FUNCTION name_sim(a text, b text, method text DEFAULT 'lev') RETURNS double precision
LANGUAGE sql IMMUTABLE AS $$
    SELECT CASE method
             WHEN 'trgm' THEN similarity(kkey(a), kkey(b))::double precision
             WHEN 'seq'  THEN seq_ratio(kkey(a), kkey(b))
             ELSE 1 - levenshtein(kkey(a), kkey(b))::double precision / greatest(length(kkey(a)), length(kkey(b)), 1)
           END
$$;
""")
# 예: 실제 명칭 변경 LFSS → LFFS를 짧은 이름만으로 비교하면 lev · seq는 0.75, trgm은 0.25
#     (짧은 문자열은 세 글자 조각이 몇 개 안 돼 trgm 값이 크게 떨어짐 → trgm이 명칭 변경을 놓치는 이유, 보고서 8.4)
print(db.execute("SELECT name_sim('LFSS', 'LFFS', 'lev'), name_sim('LFSS', 'LFFS', 'trgm'), name_sim('LFSS', 'LFFS', 'seq')").fetchone())

(0.75, 0.25, 0.75)


In [12]:
# ② SQL ⑺ edition_diff(): 판본 비교 본체 (옛 판, 새 판을 넣으면 개정 목록을 돌려줌)
# ─────────────────────────────────────────────────────────────────────────────
# Python main()의 판정 부분과 같은 순서로 처리함:
#   ① 신설 후보 _added (새 판 후보 중 옛 판 본문에 없는 것), 삭제 후보 _removed (옛 판 후보 중 새 판 본문에 없는 것)
#   ② 명칭 변경: 삭제 후보를 (절, 이름) 순으로 돌며, 같은 절의 신설 후보 중 가장 비슷한 것과 유사도가 기준 이상이면 짝지음
#      짝지은 둘은 신설 · 삭제 후보에서 뺌 (Python도 removed를 이름순으로 돌며 best를 고르고 둘 다 지움)
#   ③ 남은 신설 · 삭제를 넣음. 상대 판에 그 절 자체가 없으면 '신설 (절 신설)' · '삭제 (절 삭제)'
#   ④ 요건 변경: (절, 이름)이 두 판 모두에 있고 요건이 둘 다 있는데 req_key가 다름
#   마지막에 (절, 구분, 노테이션) 순으로 정렬 → Python CSV와 같은 줄 순서
db.execute(r"""
CREATE OR REPLACE FUNCTION edition_diff(old_doc text, new_doc text,
                                        rename_min double precision DEFAULT 0.6, method text DEFAULT 'lev')
RETURNS TABLE (section text, kind text, notation text, old_pages text, new_pages text, old_req text, new_req text)
LANGUAGE plpgsql AS $$
DECLARE
    r record;   -- 돌고 있는 삭제 후보 한 줄
    b record;   -- 그와 가장 비슷한 신설 후보 (+ 유사도 s)
BEGIN
    -- 함수 안에서만 쓰는 임시 표 (같은 연결에서 다시 부르면 지우고 새로 만듦)
    DROP TABLE IF EXISTS _added, _removed, _out;
    -- ① 신설 · 삭제 후보
    CREATE TEMP TABLE _added AS
        SELECT n.section, n.name, n.pages, n.req FROM notations n
        WHERE n.doc_id = new_doc AND absent_in(n.name, n.section, old_doc);
    CREATE TEMP TABLE _removed AS
        SELECT n.section, n.name, n.pages, n.req FROM notations n
        WHERE n.doc_id = old_doc AND absent_in(n.name, n.section, new_doc);
    CREATE TEMP TABLE _out (section text, kind text, notation text, old_pages text, new_pages text, old_req text, new_req text);

    -- ② 같은 절의 삭제 · 신설 쌍 중 이름이 비슷한 것 → 명칭 변경
    FOR r IN SELECT * FROM _removed x ORDER BY x.section, x.name LOOP
        SELECT a.*, name_sim(r.name, a.name, method) AS s INTO b
        FROM _added a WHERE a.section = r.section
        ORDER BY s DESC, a.name LIMIT 1;                   -- 가장 비슷한 것, 같으면 이름순으로 앞선 것
        IF FOUND AND b.s >= rename_min THEN
            INSERT INTO _out VALUES (r.section, '명칭 변경', r.name || ' → ' || b.name,
                                     array_to_string(r.pages, ' '), array_to_string(b.pages, ' '),
                                     coalesce(r.req, ''), coalesce(b.req, ''));
            DELETE FROM _removed x WHERE x.section = r.section AND x.name = r.name;   -- 짝지은 둘은 후보에서 뺌
            DELETE FROM _added a WHERE a.section = b.section AND a.name = b.name;
        END IF;
    END LOOP;

    -- ③ 남은 신설 (옛 판에 그 절의 후보가 하나도 없으면 절 자체가 새로 생긴 것)
    INSERT INTO _out
    SELECT a.section,
           CASE WHEN EXISTS (SELECT 1 FROM notations o WHERE o.doc_id = old_doc AND o.section = a.section)
                THEN '신설' ELSE '신설 (절 신설)' END,
           a.name, '', array_to_string(a.pages, ' '), '', coalesce(a.req, '')
    FROM _added a;

    -- ③ 남은 삭제 (새 판에 그 절의 후보가 하나도 없으면 절 자체가 없어진 것)
    INSERT INTO _out
    SELECT x.section,
           CASE WHEN EXISTS (SELECT 1 FROM notations o WHERE o.doc_id = new_doc AND o.section = x.section)
                THEN '삭제' ELSE '삭제 (절 삭제)' END,
           x.name, array_to_string(x.pages, ' '), '', coalesce(x.req, ''), ''
    FROM _removed x;

    -- ④ 이름은 같고 요건(설계 · 검사 근거)이 바뀐 경우
    INSERT INTO _out
    SELECT n.section, '요건 변경', n.name, array_to_string(o.pages, ' '), array_to_string(n.pages, ' '), o.req, n.req
    FROM notations n
    JOIN notations o ON o.doc_id = old_doc AND o.section = n.section AND o.name = n.name
    WHERE n.doc_id = new_doc
      AND coalesce(o.req, '') <> '' AND coalesce(n.req, '') <> ''
      AND req_key(o.req) <> req_key(n.req);

    RETURN QUERY SELECT * FROM _out o ORDER BY o.section, o.kind, o.notation;
END
$$;
""")
print("edition_diff() 만듦")

edition_diff() 만듦


In [13]:
# ② SQL ⑻ 비교 결과 저장 표 edition_changes · 검색용 뷰 edition_change_pages
# ─────────────────────────────────────────────────────────────────────────────
db.execute(r"""
-- 비교 결과를 저장하는 표: 새 판을 넣을 때 edition_diff()를 한 번 돌려 결과를 넣어 둠 (검색 때마다 다시 계산하지 않음)
CREATE TABLE edition_changes (
    old_doc   text NOT NULL REFERENCES documents,   -- 옛 판 'KR2022'
    new_doc   text NOT NULL REFERENCES documents,   -- 새 판 'KR2026'
    section   text NOT NULL,
    -- 구분은 네 종류만 허용 (괄호 뒤 '(절 신설)' 같은 꼬리는 빼고 검사)
    kind      text NOT NULL CHECK (split_part(kind, ' (', 1) IN ('신설', '삭제', '명칭 변경', '요건 변경')),
    notation  text NOT NULL,
    old_pages text NOT NULL,                        -- 옛 판에서 나온 쪽 (공백으로 이음, 없으면 '')
    new_pages text NOT NULL,                        -- 새 판에서 나온 쪽
    old_req   text NOT NULL,
    new_req   text NOT NULL
);

-- 검색에서 쓰는 모양: 쪽마다 붙은 개정 종류 (백엔드 part1.CHANGED와 같은 것)
--   옛 판 쪽 · 새 판 쪽을 한 줄씩 펼쳐(unnest) 'KR2026:176 | 신설' 같은 (쪽, 종류) 쌍으로 만듦
--   판 비교 질문("2026판에서 새로 생긴 …")이면 검색기가 이 쪽들을 앞으로 올림
CREATE VIEW edition_change_pages AS
    SELECT DISTINCT d.doc || ':' || p AS page_id, split_part(d.kind, ' (', 1) AS kind
    FROM (SELECT old_doc AS doc, kind, old_pages AS pg FROM edition_changes
          UNION ALL
          SELECT new_doc, kind, new_pages FROM edition_changes) d,
         unnest(string_to_array(nullif(d.pg, ''), ' ')) AS p;
""")
# 이 노트북이 만든 함수 목록 (확장 기능이 만든 함수는 빼고)
print("함수:", [r[0] for r in db.execute("""
    SELECT p.proname FROM pg_proc p JOIN pg_namespace n ON n.oid = p.pronamespace
    LEFT JOIN pg_depend d ON d.objid = p.oid AND d.deptype = 'e'
    WHERE n.nspname = 'public' AND d.objid IS NULL ORDER BY 1""")])

함수: ['absent_in', 'edition_diff', 'kkey', 'name_sim', 'req_key', 'seq_ratio']


In [14]:
# ② 파싱 결과를 DB에 넣고, 본문 뷰를 다시 계산함
# ─────────────────────────────────────────────────────────────────────────────
t = time.perf_counter()
db.execute("INSERT INTO documents VALUES ('KR2022', 'KR', 2022), ('KR2026', 'KR', 2026)")
with db.cursor() as cur:
    # COPY: 여러 줄을 한 번에 흘려 넣는 PostgreSQL 기능 (INSERT를 한 줄씩 보내는 것보다 훨씬 빠름)
    with cur.copy("COPY pages (page_id, doc_id, page_no, raw, section) FROM STDIN") as cp:
        for y in P:
            for p in sorted(P[y]):
                cp.write_row((f"KR{y}:{p}", f"KR{y}", p, P[y][p], S[y][p]))
    with cur.copy("COPY notations (doc_id, section, name, pages, req) FROM STDIN") as cp:
        for y in U:
            for (sec, name), (pgs, req) in U[y].items():
                cp.write_row((f"KR{y}", sec, name, sorted(pgs), req))   # 쪽 목록 → int[] 배열, 요건이 없으면 NULL
summary["load_s"] = round(time.perf_counter() - t, 3)

t = time.perf_counter()
db.execute("REFRESH MATERIALIZED VIEW section_texts")   # pages를 채웠으니 절 본문 · 판 본문을 다시 계산
db.execute("REFRESH MATERIALIZED VIEW doc_texts")
summary["refresh_s"] = round(time.perf_counter() - t, 3)

cnt = {tb: db.execute(f"SELECT count(*) FROM {tb}").fetchone()[0] for tb in ("documents", "pages", "notations", "section_texts")}
summary["db_rows"] = cnt
print(f"② DB 적재: {cnt} · 적재 {summary['load_s']}초 · 본문 뷰 갱신 {summary['refresh_s']}초")

② DB 적재: {'documents': 2, 'pages': 495, 'notations': 1152, 'section_texts': 92} · 적재 0.033초 · 본문 뷰 갱신 0.163초


## ④-1 비교용 정규화가 Python과 글자까지 같은지

SQL 판정이 Python과 같으려면 먼저 "비교용 열쇠"가 같아야 함. 네 가지를 대조함
- 쪽 본문: KR 두 판의 쪽 전부 (`kkey(raw)` ↔ `compact(쪽 텍스트)`)
- 노테이션 이름 전부 (`kkey(name)` ↔ `compact(name)`)
- 요건 (`req_key(req)` ↔ Python `req()`)
- 절 본문 (`section_texts.hay` ↔ Python `section_text()`)

In [15]:
# ④-1 SQL kkey · req_key · section_texts가 Python compact · req · section_text와 같은지
# ─────────────────────────────────────────────────────────────────────────────
# 쪽 본문: DB가 계산한 열쇠를 Python이 계산한 열쇠와 쪽마다 비교. page_id 'KR2026:176' → 연도 2026, 쪽 176
pg_keys = dict(db.execute("SELECT page_id, kkey(raw) FROM pages").fetchall())
bad_pages = [pid for pid, k in pg_keys.items() if k != K.compact(P[int(pid[2:6])][int(pid.split(":")[1])])]
# 노테이션 이름 · 요건
names = db.execute("SELECT name, kkey(name), req, req_key(req) FROM notations").fetchall()
bad_names = [n for n, k, _, _ in names if k != K.compact(n)]
req_fn = lambda v: K.compact(K.re.sub(r"(?<=[,\s])\d\)", "", v or ""))   # Python main() 안의 req()와 같은 식
bad_reqs = [r for _, _, r, rk in names if r is not None and rk != req_fn(r)]
# 절 본문: Python은 {절: 이어 붙인 열쇠}를 연도별로 만듦
T = {y: K.section_text(P[y], S[y]) for y in P}
sec_hay = {(d, s): h for d, s, h in db.execute("SELECT doc_id, section, hay FROM section_texts").fetchall()}
bad_secs = [(d, s) for (d, s), h in sec_hay.items() if h != T[int(d[2:])][s]]

summary["norm_mismatch"] = {"pages": len(bad_pages), "names": len(bad_names), "reqs": len(bad_reqs), "section_texts": len(bad_secs)}
print(f"④ 정규화 비교 (SQL vs Python): 쪽 {len(pg_keys)}개 중 다름 {len(bad_pages)} · 이름 {len(names)}개 중 다름 {len(bad_names)}"
      f" · 요건 다름 {len(bad_reqs)} · 절 본문 {len(sec_hay)}개 중 다름 {len(bad_secs)}")

④ 정규화 비교 (SQL vs Python): 쪽 495개 중 다름 0 · 이름 1152개 중 다름 0 · 요건 다름 0 · 절 본문 92개 중 다름 0


## ③ SQL로 판정하고 기준 93건과 비교

명칭 변경 유사도 세 가지로 각각 `edition_diff('KR2022', 'KR2026', 0.6, 방법)`을 돌림. 기준값 0.6은 Python과 같음
- **lev**: 미리 정한 주 기준 (PostgreSQL에 기본으로 있는 편집 거리)
- **trgm**: 참고
- **seq**: lev 결과를 본 뒤 추가한 것. Python과 같은 알고리즘

결과는 Python과 같은 머리글의 CSV로 저장해 바이트 비교함. "SQL에만" · "Python에만" 줄이 차이임

In [16]:
# ③ SQL 판정 세 가지(lev · trgm · seq)를 기준 93건과 비교
# ─────────────────────────────────────────────────────────────────────────────
# seq_ratio가 Python과 같으려면 신설 후보 이름의 열쇠가 200자 미만이어야 함 (Python autojunk 조건, ⑸ 셀 설명)
longest = db.execute("""SELECT max(length(kkey(name))) FROM notations
                        WHERE doc_id = 'KR2026' AND absent_in(name, section, 'KR2022')""").fetchone()[0]
summary["longest_added_name_key"] = longest   # SequenceMatcher의 b(= 신설 후보) 쪽만 autojunk 대상
assert longest < 200, "신설 후보 이름이 200자 이상이면 Python autojunk 때문에 seq가 SequenceMatcher와 달라짐"

results = {}
for method in ("lev", "trgm", "seq"):
    t = time.perf_counter()
    rows = [tuple(r) for r in db.execute("SELECT * FROM edition_diff('KR2022', 'KR2026', 0.6, %s)", (method,)).fetchall()]
    sec = round(time.perf_counter() - t, 3)
    path = OUT / f"sql_{method}_kr_edition_diff.csv"
    with open(path, "w", newline="", encoding="utf-8-sig") as f:   # Python 스크립트와 같은 형식(BOM, 줄바꿈)으로 저장
        w = csv.writer(f)
        w.writerow(HEAD)
        w.writerows(rows)
    only_sql = sorted(set(rows) - set(ref_rows))      # SQL 결과에만 있는 줄
    only_py = sorted(set(ref_rows) - set(rows))       # 기준(Python)에만 있는 줄
    kinds = defaultdict(int)                          # 구분별 건수 ('신설 (절 신설)'은 '신설'로 셈)
    for r in rows:
        kinds[r[1].split(" (")[0]] += 1
    results[method] = dict(rows=len(rows), same_bytes=md5(path) == md5(REF), same_order=rows == ref_rows,
                           only_sql=only_sql, only_python=only_py, seconds=sec, kinds=dict(kinds))
    print(f"③ SQL 판정 ({method}): {len(rows)}건 {dict(kinds)} · 기준 CSV와 바이트까지 같음 = {results[method]['same_bytes']}"
          f" · SQL에만 {len(only_sql)} · Python에만 {len(only_py)} · {sec}초")
    for r in only_sql:
        print("    SQL에만:", r[:3])         # (절, 구분, 노테이션)만 보여 줌
    for r in only_py:
        print("    Python에만:", r[:3])
summary["sql"] = results

③ SQL 판정 (lev): 94건 {'신설': 56, '요건 변경': 19, '명칭 변경': 9, '삭제': 10} · 기준 CSV와 바이트까지 같음 = False · SQL에만 2 · Python에만 1 · 0.039초
    SQL에만: ('Chemical Tanker', '삭제', 'Apparent Specific Gravity(SG) or')
    SQL에만: ('Chemical Tanker', '신설', 'Apparent Specific Gravity (SG) and/or Name of Chemical primarily carried')
    Python에만: ('Chemical Tanker', '명칭 변경', 'Apparent Specific Gravity(SG) or → Apparent Specific Gravity (SG) and/or Name of Chemical primarily carried')
③ SQL 판정 (trgm): 100건 {'신설': 62, '요건 변경': 19, '명칭 변경': 3, '삭제': 16} · 기준 CSV와 바이트까지 같음 = False · SQL에만 14 · Python에만 7 · 0.039초
    SQL에만: ('Ch 3', '삭제', 'LFSS (DF-LNG, SF-LNG) (DF-Methanol, SF-Methanol) (DF-Ethanol, SF-Ethanol)')
    SQL에만: ('Ch 3', '신설', 'LFFS (DF-LNG, SF-LNG) (DF-Methanol, SF-Methanol) (DF-Ethanol, SF-Ethanol) (DF-LPG, SF-LPG) (DF-Ammonia, SF-Ammonia)')
    SQL에만: ('Chemical Tanker', '삭제', 'Apparent Specific Gravity(SG) or')
    SQL에만: ('Chemical Tanker', '삭제', 'Name of Chemical when exclusively carried')
    

In [17]:
# ④-2 명칭 변경 후보 쌍: 같은 절의 (삭제 후보 × 신설 후보)마다 유사도를 네 가지로 계산해 비교
# ─────────────────────────────────────────────────────────────────────────────
# 판정이 갈리는 이유가 "유사도 계산 방식"인지 보려는 것. 0.6 기준에서 Python과 판정이 다른 쌍을 셈
pairs = db.execute("""
    SELECT r.section, r.name, a.name, name_sim(r.name, a.name, 'lev'), name_sim(r.name, a.name, 'trgm'),
           name_sim(r.name, a.name, 'seq')
    FROM notations r JOIN notations a ON a.section = r.section
    WHERE r.doc_id = 'KR2022' AND a.doc_id = 'KR2026'
      AND absent_in(r.name, r.section, 'KR2026') AND absent_in(a.name, a.section, 'KR2022')
    ORDER BY 1, 2, 3""").fetchall()
with open(OUT / "rename_pairs.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["절", "2022 이름", "2026 이름", "Python SequenceMatcher", "SQL levenshtein", "SQL trigram", "SQL seq_ratio",
                "0.6 판정 (py/lev/trgm/seq)"])
    flip = {"lev": 0, "trgm": 0, "seq": 0}   # Python과 0.6 판정이 다른 쌍의 수
    seq_diff = 0.0                           # seq_ratio와 Python 값의 최대 차이 (0이어야 함)
    for sec, a, b, lev, tg, sq in pairs:
        py = SequenceMatcher(None, K.compact(a), K.compact(b)).ratio()
        seq_diff = max(seq_diff, abs(py - sq))
        for m, v in (("lev", lev), ("trgm", tg), ("seq", sq)):
            flip[m] += (py >= 0.6) != (v >= 0.6)
        marks = "".join("O" if v >= 0.6 else "-" for v in (py, lev, tg, sq))   # 예) "OO-O" = trgm만 0.6 미만
        w.writerow([sec, a, b, f"{py:.3f}", f"{lev:.3f}", f"{tg:.3f}", f"{sq:.3f}", marks])
summary["rename_pairs"] = {"pairs": len(pairs), "disagree_with_python_at_0.6": flip, "seq_max_abs_diff": seq_diff}
print(f"④ 명칭 변경 후보 쌍 {len(pairs)}개: Python과 0.6 판정이 다른 쌍 {flip} · seq_ratio와 Python 값 최대 차이 {seq_diff:.2e}")

④ 명칭 변경 후보 쌍 186개: Python과 0.6 판정이 다른 쌍 {'lev': 1, 'trgm': 7, 'seq': 0} · seq_ratio와 Python 값 최대 차이 0.00e+00


In [18]:
# ⑤ 검색에 쓸 결과를 저장 표에 넣고, 검색에 쓰는 모양(쪽 → 개정 종류)이 백엔드 방식과 같은지 확인
# ─────────────────────────────────────────────────────────────────────────────
STORE = "seq" if results["seq"]["same_bytes"] else "lev"   # Python과 바이트까지 같은 seq를 저장 (아니면 주 기준 lev)
summary["stored_method"] = STORE
db.execute("TRUNCATE edition_changes")                    # 다시 실행해도 중복되지 않게 비우고 넣음
db.execute("INSERT INTO edition_changes SELECT 'KR2022', 'KR2026', * FROM edition_diff('KR2022', 'KR2026', 0.6, %s)", (STORE,))

sql_changed = defaultdict(set)                           # DB 뷰에서 읽은 {쪽: {개정 종류}}
for pid, kind in db.execute("SELECT page_id, kind FROM edition_change_pages").fetchall():
    sql_changed[pid].add(kind)
ref_changed = defaultdict(set)                           # 기준 CSV로 백엔드(part1.init)와 같은 방식으로 만든 것
with open(REF, encoding="utf-8-sig") as f:
    for r in csv.DictReader(f):
        kind = r["구분"].split(" (")[0]
        for col, doc in (("2022 쪽", "KR2022"), ("2026 쪽", "KR2026")):
            for p in r[col].split():
                ref_changed[f"{doc}:{p}"].add(kind)
summary["changed_pages"] = {"sql": len(sql_changed), "csv": len(ref_changed), "same": dict(sql_changed) == dict(ref_changed)}
print(f"⑤ 쪽 → 개정 종류 (DB 표 edition_changes, {STORE}): SQL {len(sql_changed)}쪽 · CSV {len(ref_changed)}쪽"
      f" · 같음 = {summary['changed_pages']['same']}")
for pid in sorted(set(sql_changed) | set(ref_changed)):
    if sql_changed.get(pid) != ref_changed.get(pid):
        print(f"    {pid}: SQL {sorted(sql_changed.get(pid, []))} · CSV {sorted(ref_changed.get(pid, []))}")

# 결과 저장 (집합은 정렬한 목록으로)
with open(OUT / "sql_changed_pages.json", "w", encoding="utf-8") as f:
    json.dump({k: v for k, v in sql_changed.items()}, f, ensure_ascii=False, default=sorted, indent=0)
with open(OUT / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=1, default=str)
print("저장:", OUT)

⑤ 쪽 → 개정 종류 (DB 표 edition_changes, seq): SQL 60쪽 · CSV 60쪽 · 같음 = True
저장: C:\Users\710pk\dev\hd-rag-part2\db\edition_diff\notebook_out


## ⑥ 비교표를 바꿔 끼우고 검색까지 돌려 보기

판정이 몇 건 다른 것이 검색 순위에 영향을 주는지 확인함. 에이전트 백엔드의 검색기(`rag/part1.py`, 보고서 최종 구성과 같은 코드)를 불러와, 판 비교 처리에 쓰는 표 `CHANGED`(쪽 → 개정 종류)만 다섯 가지로 바꿔 끼움

| 비교표 | 어디서 |
|---|---|
| 지금 CSV | 백엔드가 실제로 읽는 `kr_edition_diff.csv` (기준) |
| SQL seq (DB 표) | 이 DB의 `edition_change_pages` 뷰를 직접 읽음 |
| SQL lev · SQL trgm | ③에서 저장한 CSV |
| 빈 표 (대조군) | 판 비교 정보를 모두 뺌 → 이 장치가 실제로 검색에 쓰이는지 확인 |

구성 4개(E9 기준 구성 · 미리 정한 후보 · 최종 통합 E27 전 · 최종 통합 +E27) × 질문 37개(test 10 + dev 27)마다 nDCG@10과 정답 쪽 순위를 보고서 값(`final27_per_question.csv`)과 비교함
- 1단계 색인은 FAISS로 고정함(`VECTOR_DB=faiss`). Neon에 접속하지 않고 판본 비교표만 바꿔 보는 실험이라서
- 모델 호출은 막음: BGE-M3 인코딩 · 리랭커 · Groq LLM을 부르면 오류가 나게 해서, 저장된 캐시 값만으로 결과가 나왔음을 보장함 → GPU 없이 같은 결과
- 비교표는 검색 옵션 `boost`(판 비교 질문이면 그 개정 종류가 기록된 쪽을 앞으로 올림)에서만 쓰임. E9는 boost를 쓰지 않아 어떤 표든 결과가 같아야 함

In [19]:
# ⑥-1 에이전트 백엔드의 검색기를 불러옴 (모델 호출은 막고 캐시만 씀)
# ─────────────────────────────────────────────────────────────────────────────
os.environ["VECTOR_DB"] = "faiss"     # backend\.env에 Neon 주소가 있어도 FAISS(메모리) 색인을 쓰게 함. import 전에 정해야 함
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))  # backend 폴더를 import 경로에 넣어 rag 패키지를 찾게 함
from rag import part1 as R            # 검색기 모듈 (원래 스크립트에서는 P였지만, 여기서는 P가 쪽 텍스트라 R로 씀)


def blocked(*a, **k):
    """캐시에 없는 값이 필요해 모델을 부르려 하면 멈춤 → 결과가 저장된 값만으로 나왔음을 보장"""
    raise RuntimeError("캐시에 없는 값 → 모델 호출 시도")


R.M3.encode = blocked          # BGE-M3 질문 임베딩
R.cross_reranker = blocked     # bge-reranker-v2-m3 리랭커
R.groq_json = blocked          # Groq LLM (영어 검색어 만들기 등)

t = time.perf_counter()
R.init(verbose=False)          # pkl · 캐시 · FAISS 색인 2종 · 질문 세트 · 판 비교표(CHANGED) 준비
print(f"검색기 준비 {time.perf_counter() - t:.0f}초 · 1단계 색인 {R.VECTOR_DB['name']} · 질문 {len(R.QSETS)}개")

검색기 준비 1초 · 1단계 색인 FAISS · 질문 37개


In [20]:
# ⑥-2 비교표 다섯 가지 × 구성 4개 × 질문 37개를 검색해 보고서 값과 비교
# ─────────────────────────────────────────────────────────────────────────────
def from_csv(path):
    """판본 비교 CSV → {쪽 ID: {개정 종류}} (백엔드 part1.init이 CHANGED를 만드는 방식과 같음)
    '신설 (절 신설)'처럼 괄호가 붙은 구분은 앞부분('신설')만 쓰고, 옛 판 쪽 · 새 판 쪽 모두에 붙임"""
    m = defaultdict(set)
    with open(path, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            kind = r["구분"].split(" (")[0]
            for col, doc in (("2022 쪽", "KR2022"), ("2026 쪽", "KR2026")):
                for p in r[col].split():      # 쪽이 여러 개면 공백으로 이어져 있음
                    m[f"{doc}:{p}"].add(kind)
    return m


def from_db():
    """DB 뷰 edition_change_pages에서 바로 읽음 (⑤에서 seq 결과를 넣어 둔 표)"""
    m = defaultdict(set)
    for pid, kind in db.execute("SELECT page_id, kind FROM edition_change_pages"):
        m[pid].add(kind)
    return m


VARIANTS = {
    "지금 CSV": lambda: from_csv(R.OUT_DIR / "kr_edition_diff.csv"),   # 백엔드가 실제로 읽는 표 (기준)
    "SQL seq (DB 표)": from_db,
    "SQL lev": lambda: from_csv(OUT / "sql_lev_kr_edition_diff.csv"),
    "SQL trgm": lambda: from_csv(OUT / "sql_trgm_kr_edition_diff.csv"),
    "빈 표 (대조군)": lambda: defaultdict(set),
}
# 보고서의 문항별 기준 값: 질문 ID → {"구성 이름": nDCG@10, "구성 이름 순위": "1 · 2" …}
with open(FINAL27, encoding="utf-8-sig") as f:
    want = {r["qid"]: r for r in csv.DictReader(f)}
# 검색 구성 4개. idx = 1단계 색인(청킹 방식), 나머지는 search_v2의 장치 스위치
#   topic · policy · boost = 판 비교 처리(E12-4: 검색어를 주제어로, 비교 대상 판에 맞춘 구판 감점, 개정 쪽 올리기)
#   rr="both" = 리랭커 한 · 영 결합(E14 · E15),  old_last = 리랭크 뒤 구판 감점(E17)
#   FINAL_OPTS = 위 + 의도어 빼기(E16),  FINAL27_OPTS = 위 + 선급 비교 병합(E27)
CAND = dict(topic=True, policy=True, boost=True, rr="both", old_last=True)
CONFIGS = {
    "E9 기준 구성": dict(idx=R.IDX_E9),                        # struct 청킹, 장치 없음 (비교표를 쓰지 않음)
    "미리 정한 후보": dict(idx=R.IDX13, **CAND),                # 구조 기반 청킹(E13-2) + 위 장치들
    "최종 통합 (E27 전)": dict(idx=R.IDX13, **R.FINAL_OPTS),    # 제출 노트북 최종 (test 0.974)
    "최종 통합 (+E27)": dict(idx=R.IDX13, **R.FINAL27_OPTS),    # 보고서 최종 (test 0.988)
}
# 질문 37개: test(t01–t10) 먼저, 그다음 dev(d01–d27)
qs = sorted(({**v} for v in R.QSETS.values()), key=lambda q: (q["qid"][0] != "t", q["qid"]))
base = dict(R.CHANGED)      # 원래 비교표를 보관 → 끝나면 되돌림
report = {}
for vname, make in VARIANTS.items():
    m = make()
    R.CHANGED.clear()       # 검색기가 쓰는 비교표를 통째로 바꿔 끼움 (search_v2가 보는 같은 dict 객체를 고쳐야 함)
    R.CHANGED.update(m)
    changed_pages = sorted(p for p in set(m) | set(base) if m.get(p) != base.get(p))   # 기준과 개정 종류가 다른 쪽
    res = {"changed_pages_vs_csv": changed_pages}
    for cname, opts in CONFIGS.items():
        ok, nd, diffs = 0, {"t": [], "d": []}, []
        for q in qs:
            # 리랭커 한 · 영 결합(rr="both")을 쓰는 구성은 영어 검색어가 필요함 (캐시에 저장된 값)
            en = R.english_query(q["query"]) if opts.get("rr") == "both" else None
            order, _ = R.search_v2(q["query"], en=en, **opts)      # 쪽 ID 순위 목록 (50위까지 + 나머지)
            v = R.metrics_at_k(order, q["rels"])["nDCG@10"]
            nd[q["qid"][0]].append(v)
            head = order[:R.DEPTH]                                  # 50위까지
            ranks = " · ".join(str(head.index(p) + 1) if p in head else "밖" for p in q["rels"])   # 정답 쪽들의 순위
            w = want[q["qid"]]
            # 보고서 값과 같음 = nDCG@10 차이가 반올림 오차(5e-5) 안이고, 정답 쪽 순위 문자열이 같음
            same = abs(v - float(w[cname])) < 5e-5 and ranks == w[f"{cname} 순위"]
            ok += same
            if not same:
                diffs.append(f"{q['qid']}: {v:.4f} ({ranks}) vs {w[cname]} ({w[cname + ' 순위']})")
        res[cname] = {"match": f"{ok}/{len(qs)}", "test_nDCG@10": round(sum(nd["t"]) / len(nd["t"]), 4),
                      "dev_nDCG@10": round(sum(nd["d"]) / len(nd["d"]), 4), "diffs": diffs}
    report[vname] = res
    line = " | ".join(f"{c} {r['match']} (test {r['test_nDCG@10']:.4f} · dev {r['dev_nDCG@10']:.4f})"
                      for c, r in res.items() if c in CONFIGS)
    print(f"[{vname}] 기준과 개정 종류가 다른 쪽 {len(changed_pages)}개 {changed_pages}\n   {line}")
    for c in CONFIGS:
        for d in res[c]["diffs"]:
            print(f"   DIFF {c} {d}")
R.CHANGED.clear()
R.CHANGED.update(base)      # 원래 비교표로 되돌림
with open(OUT / "search_check.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=1)

[지금 CSV] 기준과 개정 종류가 다른 쪽 0개 []
   E9 기준 구성 37/37 (test 0.8230 · dev 0.9137) | 미리 정한 후보 37/37 (test 0.9444 · dev 0.9951) | 최종 통합 (E27 전) 37/37 (test 0.9743 · dev 0.9976) | 최종 통합 (+E27) 37/37 (test 0.9877 · dev 1.0000)


[SQL seq (DB 표)] 기준과 개정 종류가 다른 쪽 0개 []
   E9 기준 구성 37/37 (test 0.8230 · dev 0.9137) | 미리 정한 후보 37/37 (test 0.9444 · dev 0.9951) | 최종 통합 (E27 전) 37/37 (test 0.9743 · dev 0.9976) | 최종 통합 (+E27) 37/37 (test 0.9877 · dev 1.0000)


[SQL lev] 기준과 개정 종류가 다른 쪽 2개 ['KR2022:32', 'KR2026:32']
   E9 기준 구성 37/37 (test 0.8230 · dev 0.9137) | 미리 정한 후보 37/37 (test 0.9444 · dev 0.9951) | 최종 통합 (E27 전) 37/37 (test 0.9743 · dev 0.9976) | 최종 통합 (+E27) 37/37 (test 0.9877 · dev 1.0000)


[SQL trgm] 기준과 개정 종류가 다른 쪽 12개 ['KR2022:19', 'KR2022:224', 'KR2022:32', 'KR2022:45', 'KR2022:74', 'KR2022:85', 'KR2026:19', 'KR2026:228', 'KR2026:32', 'KR2026:45', 'KR2026:74', 'KR2026:85']
   E9 기준 구성 37/37 (test 0.8230 · dev 0.9137) | 미리 정한 후보 37/37 (test 0.9444 · dev 0.9951) | 최종 통합 (E27 전) 37/37 (test 0.9743 · dev 0.9976) | 최종 통합 (+E27) 37/37 (test 0.9877 · dev 1.0000)


[빈 표 (대조군)] 기준과 개정 종류가 다른 쪽 60개 ['KR2022:100', 'KR2022:101', 'KR2022:16', 'KR2022:17', 'KR2022:19', 'KR2022:220', 'KR2022:222', 'KR2022:223', 'KR2022:224', 'KR2022:225', 'KR2022:226', 'KR2022:227', 'KR2022:229', 'KR2022:230', 'KR2022:231', 'KR2022:32', 'KR2022:45', 'KR2022:48', 'KR2022:49', 'KR2022:74', 'KR2022:82', 'KR2022:83', 'KR2022:85', 'KR2026:100', 'KR2026:101', 'KR2026:136', 'KR2026:156', 'KR2026:17', 'KR2026:176', 'KR2026:177', 'KR2026:19', 'KR2026:204', 'KR2026:206', 'KR2026:223', 'KR2026:225', 'KR2026:226', 'KR2026:227', 'KR2026:228', 'KR2026:229', 'KR2026:230', 'KR2026:231', 'KR2026:232', 'KR2026:233', 'KR2026:234', 'KR2026:235', 'KR2026:236', 'KR2026:237', 'KR2026:238', 'KR2026:32', 'KR2026:45', 'KR2026:48', 'KR2026:49', 'KR2026:57', 'KR2026:58', 'KR2026:62', 'KR2026:66', 'KR2026:71', 'KR2026:74', 'KR2026:85', 'KR2026:93']
   E9 기준 구성 37/37 (test 0.8230 · dev 0.9137) | 미리 정한 후보 35/37 (test 0.8496 · dev 0.9951) | 최종 통합 (E27 전) 35/37 (test 0.8795 · dev 0.9976) | 최종 통합 (+E27)

## ⑦ 보고서에 쓴 결과 파일과 대조

이 노트북이 `notebook_out\`에 쓴 결과를, 보고서를 쓸 때 저장한 결과 파일(이 폴더 바로 아래)과 비교함. 모두 같으면 노트북이 원래 스크립트와 같은 계산을 했다는 뜻임
- CSV: 바이트까지 비교
- `summary.json`: 실행할 때마다 달라지는 걸린 시간을 빼고 내용 비교
- `sql_changed_pages.json`: DB가 돌려주는 순서는 정해져 있지 않아서 쪽별 종류 집합으로 비교

In [21]:
# ⑦ notebook_out\의 결과가 보고서에 쓴 결과 파일(NB_DIR 바로 아래)과 같은지
# ─────────────────────────────────────────────────────────────────────────────
same = {}
for name in ("python_kr_edition_diff.csv", "sql_lev_kr_edition_diff.csv", "sql_trgm_kr_edition_diff.csv",
             "sql_seq_kr_edition_diff.csv", "rename_pairs.csv"):
    same[name] = md5(OUT / name) == md5(NB_DIR / name)


def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def drop_times(s):
    """summary에서 실행할 때마다 달라지는 걸린 시간(…_s, seconds)을 뺌"""
    s = {k: v for k, v in s.items() if not k.endswith("_s")}
    s["sql"] = {m: {k: v for k, v in r.items() if k != "seconds"} for m, r in s["sql"].items()}
    return s


same["summary.json (걸린 시간 제외)"] = drop_times(load(OUT / "summary.json")) == drop_times(load(NB_DIR / "summary.json"))
same["search_check.json"] = load(OUT / "search_check.json") == load(NB_DIR / "search_check.json")
as_sets = lambda d: {k: sorted(v) for k, v in d.items()}
same["sql_changed_pages.json (집합)"] = as_sets(load(OUT / "sql_changed_pages.json")) == as_sets(load(NB_DIR / "sql_changed_pages.json"))
for k, v in same.items():
    print("같음 " if v else "다름!", k)
print("\n→ 모두 같음" if all(same.values()) else "\n→ 다른 파일이 있음")

같음  python_kr_edition_diff.csv
같음  sql_lev_kr_edition_diff.csv
같음  sql_trgm_kr_edition_diff.csv
같음  sql_seq_kr_edition_diff.csv
같음  rename_pairs.csv
같음  summary.json (걸린 시간 제외)
같음  search_check.json
같음  sql_changed_pages.json (집합)

→ 모두 같음


## 끝내기 · 다음에 할 일

- **서버 끄기**: 아래 셀의 주석을 풀고 실행하거나 PowerShell에서 `& "C:\Users\710pk\dev\localpg\env\Library\bin\pg_ctl.exe" -D "C:\Users\710pk\dev\localpg\data" stop`. 켜 두어도 되지만 PC를 다시 켜면 꺼져 있음 (0-2 셀이 다시 켬)
- **DB 백업 · 복원**: `localdb\hdrag.dump` (이 실험 DB를 pg_dump로 저장한 것). 새 PC나 다른 위치에서는 `localdb\restore_localdb.ps1`이 PostgreSQL 설치부터 복원까지 함
- **다음 단계** (SQL 연동 보고서 7장)
  - 백엔드가 CSV 대신 `edition_change_pages` 뷰를 직접 읽게 바꿈
  - 청크 6,698개(벡터 포함)를 이 로컬 DB에도 넣으면, 에이전트의 1단계 검색도 Neon 대신 로컬 DB로 할 수 있음 (`backend\.env`의 `DATABASE_URL`만 바꿈)

In [22]:
# 연결 닫기 (서버를 끄려면 아래 두 줄의 주석을 풀어 실행)
db.close()
# bin_dir = LOCALPG / "env" / "Library" / "bin"
# subprocess.run([str(bin_dir / "pg_ctl.exe"), "-D", str(LOCALPG / "data"), "stop", "-m", "fast"], check=True)